In [1]:
import tensorflow as tf
from tensorflow.keras import layers,models
from tensorflow.keras.models import Sequential
import os
import shutil
import kagglehub

# Download dataset
path = kagglehub.dataset_download("markdaniellampa/fish-dataset")

data_train_path = path + "/FishImgDataset/train"
data_test_path = path + "/FishImgDataset/test"
data_val_path = path + "/FishImgDataset/val"

img_width = 224
img_height = 224
batch_size = 32
epochs = 25

100%|██████████| 1.58G/1.58G [00:20<00:00, 83.0MB/s]

Extracting files...


In [2]:
train_data = tf.keras.utils.image_dataset_from_directory(
    data_train_path,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    shuffle=True
)

val_data = tf.keras.utils.image_dataset_from_directory(
    data_val_path,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    shuffle=False
)

species_name = train_data.class_names
print("Classes:", species_name)


Found 8809 files belonging to 31 classes.
Found 2751 files belonging to 31 classes.
Classes: ['Bangus', 'Big Head Carp', 'Black Spotted Barb', 'Catfish', 'Climbing Perch', 'Fourfinger Threadfin', 'Freshwater Eel', 'Glass Perchlet', 'Goby', 'Gold Fish', 'Gourami', 'Grass Carp', 'Green Spotted Puffer', 'Indian Carp', 'Indo-Pacific Tarpon', 'Jaguar Gapote', 'Janitor Fish', 'Knifefish', 'Long-Snouted Pipefish', 'Mosquito Fish', 'Mudfish', 'Mullet', 'Pangasius', 'Perch', 'Scat Fish', 'Silver Barb', 'Silver Carp', 'Silver Perch', 'Snakehead', 'Tenpounder', 'Tilapia']


In [3]:
AUTOTUNE = tf.data.AUTOTUNE

train_data = train_data.prefetch(buffer_size=AUTOTUNE)
val_data = val_data.prefetch(buffer_size=AUTOTUNE)


In [4]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])


In [5]:
import json

with open("species_class_names.json", "w") as f:
    json.dump(species_name, f)

In [6]:
import json

with open("species_class_names.json", "r") as f:
    species_name = json.load(f)


In [7]:
from tensorflow.keras.applications import MobileNetV2
inputs = layers.Input(shape=(img_height, img_width, 3))
num_classes = len(species_name)

x = data_augmentation(inputs)
x = layers.Rescaling(1./255)(x)

base_model = MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(img_height, img_width, 3)
)

base_model.trainable = False

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

species_model = models.Model(inputs, outputs)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [11]:
species_model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)


In [12]:
species_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 31)             │         3,999 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,425,951 (9.25 MB)

 Trainable params: 167,967 (656.12 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [10]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)


In [13]:
history = species_model.fit(
    train_data,
    validation_data=val_data,
    epochs=epochs,
    # callbacks=[early_stop]
)


Epoch 1/25


/usr/local/lib/python3.12/dist-packages/keras/src/backend/tensorflow/nn.py:717: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


276/276 ━━━━━━━━━━━━━━━━━━━━ 57s 171ms/step - accuracy: 0.3693 - loss: 2.3665 - val_accuracy: 0.7256 - val_loss: 0.9654
Epoch 2/25
276/276 ━━━━━━━━━━━━━━━━━━━━ 73s 153ms/step - accuracy: 0.6811 - loss: 1.1018 - val_accuracy: 0.8124 - val_loss: 0.6750
Epoch 3/25
276/276 ━━━━━━━━━━━━━━━━━━━━ 83s 156ms/step - accuracy: 0.7297 - loss: 0.9053 - val_accuracy: 0.8252 - val_loss: 0.6116
Epoch 4/25
276/276 ━━━━━━━━━━━━━━━━━━━━ 41s 150ms/step - accuracy: 0.7670 - loss: 0.7759 - val_accuracy: 0.8393 - val_loss: 0.5402
Epoch 5/25
276/276 ━━━━━━━━━━━━━━━━━━━━ 40s 145ms/step - accuracy: 0.7900 - loss: 0.6882 - val_accuracy: 0.8531 - val_loss: 0.4912
Epoch 6/25
276/276 ━━━━━━━━━━━━━━━━━━━━ 40s 145ms/step - accuracy: 0.7894 - loss: 0.6873 - val_accuracy: 0.8601 - val_loss: 0.4886
Epoch 7/25
276/276 ━━━━━━━━━━━━━━━━━━━━ 43s 154ms/step - accuracy: 0.8126 - loss: 0.6141 - val_accuracy: 0.8706 - val_loss: 0.4209
Epoch 8/25
276/276 ━━━━━━━━━━━━━━━━━━━━ 40s 144ms/step - accuracy: 0.8205 - loss: 0.5758 - val

In [14]:
import numpy as np
def predict_image(image_path):
    img = tf.keras.utils.load_img(image_path, target_size=(img_height, img_width))
    img_array = tf.keras.utils.img_to_array(img)
    # img_array = img_array / 255.0
    img_array = tf.expand_dims(img_array, 0)

    predictions = species_model.predict(img_array)
    score = tf.nn.softmax(predictions[0])

    predicted_class = species_name[np.argmax(score)]
    confidence = np.max(score) * 100

    print(f"Fish: {predicted_class}")
    print(f"Confidence: {confidence:.2f}%")

    return predicted_class, confidence

In [15]:
image = '000001.jpg'
predict_image(image)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Fish: Gold Fish
Confidence: 8.31%


('Gold Fish', np.float32(8.308025))

In [16]:
species_model.save("species_model.keras")

In [1]:
import tensorflow as tf
from tensorflow.keras import layers,models
from tensorflow.keras.models import Sequential
import os
from google.colab import drive
drive.mount('/content/drive')

dataset_path = "./drive/MyDrive/Freshwater Fish Disease Aquaculture in south asia/"
train_data_path = os.path.join(dataset_path, "Train")
test_data_path = os.path.join(dataset_path, "Test")
print(os.path.exists(train_data_path))
print(os.path.exists(test_data_path))



Mounted at /content/drive
True
True


In [2]:
img_width = 224
img_height = 224
batch_size = 32
epochs = 25

train_data = tf.keras.utils.image_dataset_from_directory(
    train_data_path,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    shuffle=True
)

val_data = tf.keras.utils.image_dataset_from_directory(
    test_data_path,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    shuffle=False
)

species_condition = train_data.class_names
print("Classes:", species_condition)


Found 2203 files belonging to 8 classes.
Found 1153 files belonging to 8 classes.
Classes: ['Bacterial Red disease', 'Bacterial diseases - Aeromoniasis', 'Bacterial gill disease', 'FreshFish', 'Fungal diseases Saprolegniasis', 'Healthy Fish', 'Parasitic diseases', 'Viral diseases White tail disease']


In [3]:
from collections import Counter
print(Counter(train_data_path))

Counter({'Bacterial Red disease': 1, 'Bacterial diseases - Aeromoniasis': 1, 'Bacterial gill disease': 1, 'FreshFish': 1, 'Fungal diseases Saprolegniasis': 1, 'Healthy Fish': 1, 'Parasitic diseases': 1, 'Viral diseases White tail disease': 1})


In [5]:
AUTOTUNE = tf.data.AUTOTUNE

train_data = train_data.prefetch(buffer_size=AUTOTUNE)
val_data = val_data.prefetch(buffer_size=AUTOTUNE)

In [6]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])


In [7]:
import json

with open("health_class_names.json", "w") as f:
    json.dump(species_condition, f)

In [8]:
import json

with open("health_class_names.json", "r") as f:
    species_name = json.load(f)


In [13]:
from tensorflow.keras.applications import MobileNetV2
inputs = layers.Input(shape=(img_height, img_width, 3))
num_classes = len(species_condition)

x = data_augmentation(inputs)
x = layers.Rescaling(1./255)(x)

base_model = MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(img_height, img_width, 3)
)

base_model.trainable = False

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

health_model = models.Model(inputs, outputs)

In [15]:
health_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [16]:
health_model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_1 (Sequential)       │ (None, 7, 7, 1280)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_2 (Rescaling)         │ (None, 7, 7, 1280)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 8)              │         1,032 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,984 (9.24 MB)

 Trainable params: 165,000 (644.53 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [17]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [18]:
health_model.fit(
    train_data,
    validation_data=val_data,
    epochs=epochs,
    callbacks=[early_stop]
)

Epoch 1/25
69/69 ━━━━━━━━━━━━━━━━━━━━ 523s 7s/step - accuracy: 0.1527 - loss: 2.3571 - val_accuracy: 0.3955 - val_loss: 2.0708
Epoch 2/25
69/69 ━━━━━━━━━━━━━━━━━━━━ 13s 187ms/step - accuracy: 0.2073 - loss: 2.0759 - val_accuracy: 0.3955 - val_loss: 2.0557
Epoch 3/25
69/69 ━━━━━━━━━━━━━━━━━━━━ 14s 196ms/step - accuracy: 0.2071 - loss: 2.0716 - val_accuracy: 0.3955 - val_loss: 2.0411
Epoch 4/25
69/69 ━━━━━━━━━━━━━━━━━━━━ 13s 188ms/step - accuracy: 0.2122 - loss: 2.0672 - val_accuracy: 0.3955 - val_loss: 2.0284
Epoch 5/25
69/69 ━━━━━━━━━━━━━━━━━━━━ 20s 182ms/step - accuracy: 0.2116 - loss: 2.0640 - val_accuracy: 0.3955 - val_loss: 2.0170
Epoch 6/25
69/69 ━━━━━━━━━━━━━━━━━━━━ 13s 191ms/step - accuracy: 0.2105 - loss: 2.0616 - val_accuracy: 0.3955 - val_loss: 2.0070
Epoch 7/25
69/69 ━━━━━━━━━━━━━━━━━━━━ 13s 184ms/step - accuracy: 0.2112 - loss: 2.0591 - val_accuracy: 0.3955 - val_loss: 1.9973
Epoch 8/25
69/69 ━━━━━━━━━━━━━━━━━━━━ 13s 193ms/step - accuracy: 0.2199 - loss: 2.0544 - val_accura

In [28]:
import tensorflow as tf
def predict_health(image_path):
    img = tf.keras.utils.load_img(image_path, target_size=(img_height, img_width))
    img_array = tf.keras.utils.img_to_array(img)
    # img_array = img_array / 255.0
    img_array = tf.expand_dims(img_array, 0)

    predictions = health_model.predict(img_array)
    score = tf.nn.softmax(predictions[0])

    predicted_class = species_name[np.argmax(score)]
    confidence = np.max(score) * 100

    print(f"Fish: {predicted_class}")
    print(f"Confidence: {confidence:.2f}%")

    return predicted_class, confidence

In [31]:
import numpy as np
img_width = 224
img_height = 224
batch_size = 32
epochs = 25

image = '000001.jpg'
predict_health(image)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Fish: Parasitic diseases
Confidence: 22.80%


('Parasitic diseases', np.float32(22.803396))

In [32]:
health_model.save("health_model.keras")